# TradeSense · Component 03 · notebook 55: are the explanations right about the area? Group level, top-2 and "what if normal" (protocol E3 v1; CPU, minutes)

**Question.** Notebook 51 reported the feature-level faithfulness hypotheses H-C5 and H-C9 as FAILED (hit rates 36.5% and 48.9%). This notebook tests, under a new pre-registered protocol and on held-out TEST events, three follow-up questions: is the explanation right about the *group* of features, is the top feature among the *two* that matter most, and does a direct "what if this feature were normal" explanation point at the injected cause at least as well. The results of notebook 51 are not edited.

**Order of work.** (1) protocol hash, (2) frozen models and TEST events, (3) functions and events, (4) unit tests, (5) results. Nothing is retrained, nothing is tuned: the methods have no fitted parameters.

**Protocol text hashed by this notebook:**

```
# Protocol E3 v1 (go-ahead given by the user 2026-10-18; frozen when notebook 55 hashes this exact text) - explanations at group level, top-2 agreement and counterfactual ("what if normal") explanations (novelty 3, follow-up to protocol E2 v1b)
Written BEFORE any number of this analysis exists. Nothing frozen (models, 6 features, normalisation, fusion stacker, thresholds q) changes. No new model, no training, CPU. Protocol E2 v1b and its notebook 51 results (H-C5 and H-C9 FAILED at feature level: top-1 hit 36.5% XGBoost, 48.9% Transformer) stay as reported and are not edited.

## Why the feature-level test failed (diagnosis, no new number)
(a) The six features come in correlated pairs (a trade-flow imbalance rarely moves without rush-order density; large trades move with volume acceleration; price move with range). TreeSHAP shares credit across correlated columns, while deleting one feature leaves its partner in place, so a top-1 test at the level of a single feature is stricter than what an analyst needs. (b) "Is the top explanation exactly the single largest drop" is also stricter than "does the explanation point at the right area". Two questions follow, and one product question: are the explanations right about the GROUP of features, is the top feature among the two that matter most, and would a direct what-if (counterfactual) explanation point at the injected cause at least as well?

## Data and events (held-out for explanations)
Saved TEST-B scores and the TEST-3 events (replica 0) of September and October 2024 (event start t0 >= 44,640). The explanation methods have no fitted parameters, so nothing is selected or tuned on any data; this is the first time explanations are evaluated on these events (TEST scores were opened once in T0; as in notebook 49 this is a NEW pre-registered analysis on saved scores and recomputed explanations). Event definition and model-active rule exactly as in E2: explained bar = bar of highest saved score of that model inside the event; MODEL-ACTIVE if that score is above the model's own rolling alert line (budget 2, declared q); only model-active events enter the hypotheses of that model; excluded counts are reported. Models, nets, normalisation and the deletion operation (set the N-normalised value to 0 over the last 64 bars) are those of E2.

## Declared groups (fixed now, from how the features are built, not from data)
price = {log_returns, hl_spread_ratio}; flow = {trade_flow_imbalance, rush_order_density}; size_volume = {large_trade_count, volume_acceleration}.
- Group attribution = sum of the model attribution (XGBoost: TreeSHAP shares summed to base features; Transformer: sum over the window of |gradient x input|) over the group's features.
- Group deletion D_g = output before minus output after deleting BOTH features of the group together.
- Counterfactual ("what if normal") explanation: attribution_j := D_j (single-feature deletion). It is exact for the deletion operation by construction, so it is NOT tested for faithfulness to deletion; it is tested against the injected truth (below).

## Hypotheses (each for XGBoost and for the Transformer on its model-active TEST events; xgb = x, transformer = t)
- H-E1 (group faithfulness): share of events whose top group by attribution equals the group with the largest D_g >= 0.60 (chance 1/3), AND the event-bootstrap 95% lower bound of that share > 1/3.
- H-E2 (top-2 agreement): share of events whose top-1 attribution feature lies among the two features with the largest single-feature D >= 0.65 (chance 1/3), AND the lower bound > 1/3.
- H-E3 (what-if points at the injected cause): the share of events whose top-1 counterfactual feature (largest D_j) lies in the family's expected signature (declared in notebook 45) is >= (event-weighted chance = signature size / 6) + 0.15, AND is not worse than the model-attribution top-1 by more than 0.03 (paired event bootstrap, 95% lower bound of the difference > -0.03).
- H-E4 (exactness, one hypothesis): SHAP additivity <= 1e-3; recomputed XGBoost margin equals the saved score <= 1e-3; recomputed Transformer output equals the saved score <= 5e-3 (device difference, as in E2 amendment 1); gradient first-order check within 5%; counterfactual deletion touches one column only.
- Descriptive (no decision): the E2 feature-level top-1 hit and Spearman on these TEST events (replication), per family tables, random-attribution baselines for every measure, mean drop per group, share of events where the model-attribution top-1 equals the counterfactual top-1.

## Product rules declared now (so the result cannot change them afterwards)
- If H-E1 holds for a model: the dashboard may show the GROUP-level reason for that model ("mainly price movement") as the main reason; feature names stay "indicative" and are listed as contributors, never as a single cause.
- If H-E3 holds for a model: the dashboard adds, for flagged minutes, a "what if this feature had been normal" column (the measured drop of the model score) next to the attribution.
- If a hypothesis fails: the wording stays "indicative" for that model and the failure is reported next to it.

## Known weaknesses, stated in advance
Zero-replacement measures marginal importance; group deletion removes two features at once and can exceed the sum of the single drops; the injected signatures are synthetic and generous (they list several features per family); explanations describe the model's score, not the cause of a real market event. Failed hypotheses are reported as failed and nothing is retuned.

```


In [4]:
import os, sys, json, glob, time, hashlib, warnings, datetime
from pathlib import Path
import numpy as np, pandas as pd
from scipy.stats import spearmanr
warnings.filterwarnings('ignore')
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not in Colab -> using TRADESENSE_BASE.')
import torch, xgboost as xgb
BASE = Path(os.environ.get('TRADESENSE_BASE', '/content/drive/MyDrive/research datasets'))
L3 = BASE/'layer3_v3'; OUT = BASE/'models_b2_v1'; SYMBOLS = ['BTCUSDT', 'ETHUSDT']
MOCK = os.environ.get('TS_MOCK') == '1'; NBOOT = 300 if MOCK else 3000
LINES = []
def say(s=''): print(s); LINES.append(str(s))
def sha_file(p, n=16): return hashlib.sha256(Path(p).read_bytes()).hexdigest()[:n]
hf = L3/'protocol_bar_models_v2.sha256'; pf = L3/'protocol_bar_models_v2.md'
assert hf.exists() and pf.exists(), 'protocol B2 missing'
ph = hashlib.sha256(pf.read_text().encode()).hexdigest(); assert ph == hf.read_text().split()[0], 'frozen protocol B2 was changed'
PROTO_E3 = r"""# Protocol E3 v1 (go-ahead given by the user 2026-10-18; frozen when notebook 55 hashes this exact text) - explanations at group level, top-2 agreement and counterfactual ("what if normal") explanations (novelty 3, follow-up to protocol E2 v1b)
Written BEFORE any number of this analysis exists. Nothing frozen (models, 6 features, normalisation, fusion stacker, thresholds q) changes. No new model, no training, CPU. Protocol E2 v1b and its notebook 51 results (H-C5 and H-C9 FAILED at feature level: top-1 hit 36.5% XGBoost, 48.9% Transformer) stay as reported and are not edited.

## Why the feature-level test failed (diagnosis, no new number)
(a) The six features come in correlated pairs (a trade-flow imbalance rarely moves without rush-order density; large trades move with volume acceleration; price move with range). TreeSHAP shares credit across correlated columns, while deleting one feature leaves its partner in place, so a top-1 test at the level of a single feature is stricter than what an analyst needs. (b) "Is the top explanation exactly the single largest drop" is also stricter than "does the explanation point at the right area". Two questions follow, and one product question: are the explanations right about the GROUP of features, is the top feature among the two that matter most, and would a direct what-if (counterfactual) explanation point at the injected cause at least as well?

## Data and events (held-out for explanations)
Saved TEST-B scores and the TEST-3 events (replica 0) of September and October 2024 (event start t0 >= 44,640). The explanation methods have no fitted parameters, so nothing is selected or tuned on any data; this is the first time explanations are evaluated on these events (TEST scores were opened once in T0; as in notebook 49 this is a NEW pre-registered analysis on saved scores and recomputed explanations). Event definition and model-active rule exactly as in E2: explained bar = bar of highest saved score of that model inside the event; MODEL-ACTIVE if that score is above the model's own rolling alert line (budget 2, declared q); only model-active events enter the hypotheses of that model; excluded counts are reported. Models, nets, normalisation and the deletion operation (set the N-normalised value to 0 over the last 64 bars) are those of E2.

## Declared groups (fixed now, from how the features are built, not from data)
price = {log_returns, hl_spread_ratio}; flow = {trade_flow_imbalance, rush_order_density}; size_volume = {large_trade_count, volume_acceleration}.
- Group attribution = sum of the model attribution (XGBoost: TreeSHAP shares summed to base features; Transformer: sum over the window of |gradient x input|) over the group's features.
- Group deletion D_g = output before minus output after deleting BOTH features of the group together.
- Counterfactual ("what if normal") explanation: attribution_j := D_j (single-feature deletion). It is exact for the deletion operation by construction, so it is NOT tested for faithfulness to deletion; it is tested against the injected truth (below).

## Hypotheses (each for XGBoost and for the Transformer on its model-active TEST events; xgb = x, transformer = t)
- H-E1 (group faithfulness): share of events whose top group by attribution equals the group with the largest D_g >= 0.60 (chance 1/3), AND the event-bootstrap 95% lower bound of that share > 1/3.
- H-E2 (top-2 agreement): share of events whose top-1 attribution feature lies among the two features with the largest single-feature D >= 0.65 (chance 1/3), AND the lower bound > 1/3.
- H-E3 (what-if points at the injected cause): the share of events whose top-1 counterfactual feature (largest D_j) lies in the family's expected signature (declared in notebook 45) is >= (event-weighted chance = signature size / 6) + 0.15, AND is not worse than the model-attribution top-1 by more than 0.03 (paired event bootstrap, 95% lower bound of the difference > -0.03).
- H-E4 (exactness, one hypothesis): SHAP additivity <= 1e-3; recomputed XGBoost margin equals the saved score <= 1e-3; recomputed Transformer output equals the saved score <= 5e-3 (device difference, as in E2 amendment 1); gradient first-order check within 5%; counterfactual deletion touches one column only.
- Descriptive (no decision): the E2 feature-level top-1 hit and Spearman on these TEST events (replication), per family tables, random-attribution baselines for every measure, mean drop per group, share of events where the model-attribution top-1 equals the counterfactual top-1.

## Product rules declared now (so the result cannot change them afterwards)
- If H-E1 holds for a model: the dashboard may show the GROUP-level reason for that model ("mainly price movement") as the main reason; feature names stay "indicative" and are listed as contributors, never as a single cause.
- If H-E3 holds for a model: the dashboard adds, for flagged minutes, a "what if this feature had been normal" column (the measured drop of the model score) next to the attribution.
- If a hypothesis fails: the wording stays "indicative" for that model and the failure is reported next to it.

## Known weaknesses, stated in advance
Zero-replacement measures marginal importance; group deletion removes two features at once and can exceed the sum of the single drops; the injected signatures are synthetic and generous (they list several features per family); explanations describe the model's score, not the cause of a real market event. Failed hypotheses are reported as failed and nothing is retuned.
"""
PHE = hashlib.sha256(PROTO_E3.encode()).hexdigest(); pp = OUT/'protocol_e3_v1.md'
if pp.exists(): assert pp.read_text() == PROTO_E3, 'protocol E3 exists and differs from this notebook'
else: pp.write_text(PROTO_E3)
(OUT/'protocol_e3_v1.sha256').write_text(PHE + '  protocol_e3_v1.md\n'); assert (OUT/'declared_explain_v2.json').exists(), 'run notebook 51 first'
for f in ['declared_final_set_v1.json', 'declared_ae_v2.json', 'declared_explain_v1.json', 'test_b_results_v1.json', 'b2_lib.py', 'c03_infer.py']: assert (OUT/f).exists(), f'{f} missing'
sys.path.insert(0, str(OUT)); import b2_lib as B; import c03_infer as c
torch.set_num_threads(2)
say('protocol E3 v1 sha256 ' + PHE[:16] + '; protocol B2 intact ' + ph[:16] + '; library ' + sha_file(OUT/'c03_infer.py') + '; xgboost ' + xgb.__version__ + ', torch ' + torch.__version__ + ('  [MOCK RUN]' if MOCK else ''))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
protocol E3 v1 sha256 8b71b908e324b864; protocol B2 intact 443f99d5faf5929f; library 3b5d9df23521adf9; xgboost 3.4.1, torch 2.11.0+cpu


## 1 · Frozen models, TEST features and events

In [5]:
AOFULL = 132480; AOLEN = AOFULL; TEST_LO = 31 * 1440; TAGM = dict(zip(c.MODEL_NAMES, ['AE', 'XGB', 'TR'])); WARM = B.WARM; DELLEN = 64; PRE = 130
T32 = json.load(open(OUT/'test_b_results_v1.json')); DF = json.load(open(OUT/'declared_final_set_v1.json')); MODELS = c.MODEL_NAMES; MODES = {m: DF['models'][m]['normalisation'] for m in MODELS}; FILES = {MODELS[0]: 'ae_ctx_agg', MODELS[1]: 'xgb_v2', MODELS[2]: 'transformer_v2'}
SIGN = {1: ['log_returns', 'trade_flow_imbalance', 'hl_spread_ratio', 'volume_acceleration', 'large_trade_count'], 2: ['log_returns', 'trade_flow_imbalance', 'hl_spread_ratio', 'volume_acceleration', 'large_trade_count'],
        3: ['volume_acceleration', 'large_trade_count', 'rush_order_density'], 4: ['trade_flow_imbalance', 'log_returns'], 5: ['rush_order_density', 'hl_spread_ratio', 'trade_flow_imbalance'],
        6: ['hl_spread_ratio', 'log_returns', 'large_trade_count'], 7: ['log_returns', 'volume_acceleration', 'large_trade_count', 'trade_flow_imbalance', 'hl_spread_ratio'], 8: ['trade_flow_imbalance', 'log_returns', 'large_trade_count', 'volume_acceleration']}
FEATS = B.FEATS; BASEOF = [n.split('__')[0] for n in c.FA_NAMES]; GROUPS = FEATS + ['x']; GIDX = {g_: [i for i, n in enumerate(BASEOF) if n == g_] for g_ in GROUPS}; assert sum(len(v) for v in GIDX.values()) == 53
def find1(pat):
    cc = [x for x in glob.glob(str(BASE/'**'/pat), recursive=True) if 'backup' not in x.lower()]; assert cc, pat; return sorted(cc)[0]
N0 = {}; AEC = {}; BUN = {}; FR = {}; EV = {}; SAV = {}; THR = {}
for s in SYMBOLS:
    a = pd.read_parquet(find1(f'{s.lower()}_l1_feature_matrix_90d.parquet')); a = a.set_index('datetime') if 'datetime' in a.columns else a
    N0[s] = B.fit_n0(a[FEATS].reset_index(drop=True).iloc[15:90720]); AEC[s] = np.load(OUT/'scores'/f'ae_ctx_{s}_N1.npz'); BUN[s] = c.load_bundle_from_training(OUT, s, DF, AEC[s]['scale'], N0[s], 'cpu')
    f = pd.read_parquet(L3/f'l3v3_{s}_TEST.parquet'); FR[s] = f[f.replica == 0].reset_index(drop=True).iloc[:AOLEN][FEATS]; e = pd.read_csv(L3/f'l3v3_{s}_TEST_events.csv'); EV[s] = e[e.replica == 0].reset_index(drop=True); EV[s] = EV[s][EV[s].t0 >= TEST_LO].reset_index(drop=True)
    assert sha_file(OUT/'scores'/f'test_b_{s}.npz') == T32['test_score_files_sha16'][s], 'saved TEST scores differ from what notebook 32 recorded'; z = np.load(OUT/'scores'/f'test_b_{s}.npz')
    for m in MODELS[1:]:
        SAV[(m, s)] = c.nanw(z[TAGM[m] + '__r0'][:AOLEN]); THR[(m, s)] = c.rolling_threshold(c.nanw(z[TAGM[m] + '__GEN'][:AOLEN]), DF['models'][m]['q_T1_by_budget']['2'][s])
Z = {(m, s): np.asarray(B.normalise(FR[s], MODES[m], N0[s]), 'float64') for s in SYMBOLS for m in MODELS[1:]}
say('TEST events per symbol (Sep and Oct only) loaded; frozen models loaded; declared XGBoost ' + DF['models'][MODELS[1]]['configuration'] + ' (' + MODES[MODELS[1]] + '), Transformer ' + DF['models'][MODELS[2]]['configuration'] + ' (' + MODES[MODELS[2]] + '); CAL-3 events per symbol ' + str({s: len(EV[s]) for s in SYMBOLS}))

TEST events per symbol (Sep and Oct only) loaded; frozen models loaded; declared XGBoost A|pooled|d3 (N0), Transformer W16_d32_L1 (N1); CAL-3 events per symbol {'BTCUSDT': 300, 'ETHUSDT': 300}


## 2 · Functions (from notebook 51 plus groups)

In [6]:
def xgb_x(s, z, b):
    """53 features of the last row of a short normalised slice ending at bar b"""
    return c.build_feats(z[b - 120:b + 1])[0][-1:]
def xgb_margin(s, X):
    dm = xgb.DMatrix(X, feature_names=c.FA_NAMES); return float(np.mean([b_.predict(dm, output_margin=True, iteration_range=(0, bi + 1)) for b_, bi in BUN[s].xgb], axis=0)[0])
def xgb_shap(s, X):
    dm = xgb.DMatrix(X, feature_names=c.FA_NAMES); con = np.mean([b_.predict(dm, pred_contribs=True, iteration_range=(0, bi + 1)) for b_, bi in BUN[s].xgb], axis=0)[0]; return con[:-1], con[-1]
def shap6(con): return np.array([con[GIDX[g_]].sum() for g_ in FEATS])
def delete_xgb(s, z, b, j):
    zz = z[b - 120:b + 1].copy(); zz[-DELLEN:, j] = 0.0; return c.build_feats(zz)[0][-1:]
def tr_out(s, win):
    with torch.no_grad(): x = torch.tensor(np.asarray(win, 'float32')[None]); return float(np.mean([n_(x).float().item() for n_ in BUN[s].tr_nets]))
def tr_gradxinput(s, win):
    """mean over the declared nets of d output / d input, times input; returns (W, 6) and the mean gradient"""
    x = torch.tensor(np.asarray(win, 'float32')[None]); x.requires_grad_(True); y = 0.0
    for n_ in BUN[s].tr_nets: y = y + n_(x).sum() / len(BUN[s].tr_nets)
    y.backward(); g = x.grad[0].numpy().astype('float64'); return g * np.asarray(win, 'float64'), g
def delete_tr(win, j):
    w2 = np.asarray(win, 'float32').copy(); w2[-DELLEN:, j] = 0.0; return w2
def explain_tgnn(*args, **kwargs):
    """Interface for the TGNN explanation. Nothing is produced until the TGNN is integrated and tested."""
    return {'available': False, 'reason': 'TGNN is not integrated yet; no explanation is produced or claimed', 'method': None, 'feature_attribution': None, 'time_attribution': None, 'faithfulness_tested': False}
# ---------------- new in E3: declared groups, group deletion, counterfactual attribution
GROUPS3 = {'price': ['log_returns', 'hl_spread_ratio'], 'flow': ['trade_flow_imbalance', 'rush_order_density'], 'size_volume': ['large_trade_count', 'volume_acceleration']}
GN = list(GROUPS3); GI = [[FEATS.index(f_) for f_ in GROUPS3[g_]] for g_ in GN]; assert sorted(sum(GI, [])) == list(range(6))
def delete_xgb_set(z, b, cols):
    zz = z[b - 120:b + 1].copy(); zz[-DELLEN:, cols] = 0.0; return c.build_feats(zz)[0][-1:]
def delete_tr_set(win, cols):
    w2 = np.asarray(win, 'float32').copy(); w2[-DELLEN:, cols] = 0.0; return w2
def grp(v6): return np.array([float(np.sum(np.asarray(v6)[ix])) for ix in GI])


## 3 · Model-active events

In [7]:
def peak_bar(s, m, ev):
    sc = SAV[(m, s)]; ii = np.arange(ev.t0, ev.t0 + ev.dur); v = sc[ii]; v = np.where(np.isfinite(v), v, -np.inf); return int(ii[int(np.argmax(v))]) if np.isfinite(v).any() and v.max() > -np.inf else None
XE = []; TE = []; excl = {'XGBoost v2': 0, 'Transformer v2': 0}; W_ = BUN[SYMBOLS[0]].tr_W
for s in SYMBOLS:
    for ev in EV[s].itertuples():
        for m, store in [(MODELS[1], XE), (MODELS[2], TE)]:
            b = peak_bar(s, m, ev)
            if b is None or b < PRE: excl[m] += 1; continue
            if not (SAV[(m, s)][b] > THR[(m, s)][b]): excl[m] += 1; continue
            store.append((s, ev.family, ev.amp, b))
say(f'model-active events: XGBoost {len(XE)} (excluded {excl[MODELS[1]]}), Transformer {len(TE)} (excluded {excl[MODELS[2]]}) of {sum(len(v) for v in EV.values())}; Transformer window {W_} bars')

model-active events: XGBoost 431 (excluded 169), Transformer 412 (excluded 188) of 600; Transformer window 16 bars


## 4 · Unit tests

In [8]:
s0 = SYMBOLS[0]; s_, f_, a_, b_ = XE[0] if XE else (s0, 1, 0, 2000); zx = Z[(MODELS[1], s_)]; X = xgb_x(s_, zx, b_); con, bias = xgb_shap(s_, X)
assert abs(con.sum() + bias - xgb_margin(s_, X)) <= 1e-3; say('PASS T01 SHAP additivity: contributions + bias equal the booster margin (first event)')
errs = []
for (s_, f_, a_, b_) in XE[:20]: errs.append(abs(xgb_margin(s_, xgb_x(s_, Z[(MODELS[1], s_)], b_)) - SAV[(MODELS[1], s_)][b_]))
assert max(errs) <= 1e-3, max(errs); say(f'PASS T02 XGBoost margin recomputed from a short slice equals the saved score on {len(errs)} event peak bars (max difference {max(errs):.1e})')
errs = []
for (s_, f_, a_, b_) in TE[:20]: errs.append(abs(tr_out(s_, Z[(MODELS[2], s_)][b_ - W_ + 1:b_ + 1]) - SAV[(MODELS[2], s_)][b_]))
assert max(errs) <= 5e-3, max(errs); say(f'PASS T03 Transformer output recomputed here equals the saved score on {len(errs)} event peak bars (max difference {max(errs):.1e}; tolerance 5e-3 because the saved scores come from another device, protocol amendment 1)')
rg = np.random.default_rng(0); rel = []
for (s_, f_, a_, b_) in TE[:20]:
    win = Z[(MODELS[2], s_)][b_ - W_ + 1:b_ + 1]; ai, g = tr_gradxinput(s_, win); d = rg.normal(size=win.shape) * 1e-2; lin = float((g * d).sum()); act = tr_out(s_, win + d) - tr_out(s_, win)
    if abs(lin) > 1e-6: rel.append(abs(act - lin) / abs(lin))
assert len(rel) and np.median(rel) <= 0.05, np.median(rel); say(f'PASS T04 gradient first-order check: median relative error {np.median(rel):.3f} over {len(rel)} windows')
win = Z[(MODELS[2], s_)][b_ - W_ + 1:b_ + 1] if TE else np.zeros((W_, 6)); w2 = delete_tr(win, 2); assert np.allclose(np.delete(w2, 2, axis=1), np.delete(win.astype('float32'), 2, axis=1)) and (w2[-DELLEN:, 2] == 0).all(); say('PASS T05 deleting feature j changes only column j (and only the last 64 bars)')
tg = explain_tgnn(); assert tg['available'] is False and tg['feature_attribution'] is None and tg['time_attribution'] is None; say('PASS T06 the TGNN explanation interface returns available = false and no numbers')
# T07 group deletion changes exactly the group's two columns; groups partition the six features
z_ = np.arange(40 * 6, dtype='float32').reshape(40, 6) + 1; w_ = delete_tr_set(z_, GI[1]); assert (w_[-DELLEN:, GI[1]] == 0).all() and np.array_equal(np.delete(w_, GI[1], axis=1), np.delete(z_, GI[1], axis=1)); assert np.allclose(grp(np.arange(6.)), [sum(np.arange(6.)[ix]) for ix in GI]); say('PASS T07 group deletion touches only the two columns of the group; the three groups partition the six features')
# T08 deleting nothing leaves the output unchanged, deleting all six equals the all-normal window
if TE: s_, f_, a_, b_ = TE[0]; win = Z[(MODELS[2], s_)][b_ - W_ + 1:b_ + 1]; assert abs(tr_out(s_, delete_tr_set(win, [])) - tr_out(s_, win)) < 1e-6; say('PASS T08 empty deletion leaves the Transformer output unchanged')


PASS T01 SHAP additivity: contributions + bias equal the booster margin (first event)
PASS T02 XGBoost margin recomputed from a short slice equals the saved score on 20 event peak bars (max difference 0.0e+00)
PASS T03 Transformer output recomputed here equals the saved score on 20 event peak bars (max difference 1.1e-06; tolerance 5e-3 because the saved scores come from another device, protocol amendment 1)
PASS T04 gradient first-order check: median relative error 0.019 over 20 windows
PASS T05 deleting feature j changes only column j (and only the last 64 bars)
PASS T06 the TGNN explanation interface returns available = false and no numbers
PASS T07 group deletion touches only the two columns of the group; the three groups partition the six features
PASS T08 empty deletion leaves the Transformer output unchanged


## 5 · Results (protocol E3 v1)

In [9]:

def spear(a, b):
    if np.std(a) == 0 or np.std(b) == 0: return 0.0
    return float(spearmanr(a, b)[0])
rg = np.random.default_rng(7); rows = []
for m, store in [(MODELS[1], XE), (MODELS[2], TE)]:
    for (s, fam, amp, b) in store:
        if m == MODELS[1]:
            z = Z[(m, s)]; X = xgb_x(s, z, b); con, bias = xgb_shap(s, X); at6 = shap6(con); o0 = xgb_margin(s, X)
            D6 = np.array([o0 - xgb_margin(s, delete_xgb(s, z, b, j)) for j in range(6)]); DG = np.array([o0 - xgb_margin(s, delete_xgb_set(z, b, ix)) for ix in GI])
        else:
            z = Z[(m, s)]; win = z[b - W_ + 1:b + 1]; ai, g = tr_gradxinput(s, win); at6 = np.abs(ai).sum(axis=0); o0 = tr_out(s, win)
            D6 = np.array([o0 - tr_out(s, delete_tr(win, j)) for j in range(6)]); DG = np.array([o0 - tr_out(s, delete_tr_set(win, ix)) for ix in GI])
        rows.append({'model': m, 'sym': s, 'family': int(fam), 'attr': at6, 'D': D6, 'DG': DG, 'rand': rg.permutation(6).astype(float), 'randg': rg.permutation(3).astype(float)})
R = pd.DataFrame(rows)
def boot(x, n=NBOOT, seed=0):
    x = np.asarray(x, float); r_ = np.random.default_rng(seed); return np.percentile([x[r_.integers(0, len(x), len(x))].mean() for _ in range(n)], [2.5, 97.5])
def boot_diff(a, b_, n=NBOOT, seed=0):
    d = np.asarray(a, float) - np.asarray(b_, float); r_ = np.random.default_rng(seed); return np.percentile([d[r_.integers(0, len(d), len(d))].mean() for _ in range(n)], [2.5, 97.5])
OUTR = {}; tab = []
for m in [MODELS[1], MODELS[2]]:
    r = R[R.model == m]
    if len(r) < 10: OUTR[m] = None; continue
    top1 = np.array([float(np.argmax(a) == np.argmax(d)) for a, d in zip(r.attr, r.D)]); rho = np.array([spear(a, d) for a, d in zip(r.attr, r.D)])
    top2 = np.array([float(np.argmax(a) in np.argsort(-d)[:2]) for a, d in zip(r.attr, r.D)]); top2r = np.array([float(np.argmax(a) in np.argsort(-d)[:2]) for a, d in zip(r.rand, r.D)])
    gh = np.array([float(np.argmax(grp(a)) == np.argmax(d)) for a, d in zip(r.attr, r.DG)]); ghr = np.array([float(np.argmax(a) == np.argmax(d)) for a, d in zip(r.randg, r.DG)])
    sg = [SIGN[int(f)] for f in r.family]; chance_sig = np.array([len(x) / 6 for x in sg])
    occ = np.array([float(FEATS[int(np.argmax(d))] in x) for d, x in zip(r.D, sg)]); att = np.array([float(FEATS[int(np.argmax(a))] in x) for a, x in zip(r.attr, sg)])
    same = np.array([float(np.argmax(a) == np.argmax(d)) for a, d in zip(r.attr, r.D)])
    cg, c2, co, cd = boot(gh), boot(top2), boot(occ), boot_diff(occ, att)
    OUTR[m] = {'n': len(r), 'feature_top1': float(top1.mean()), 'feature_rho': float(rho.mean()), 'group_hit': float(gh.mean()), 'group_ci': [float(x) for x in cg], 'group_random': float(ghr.mean()), 'top2': float(top2.mean()), 'top2_ci': [float(x) for x in c2], 'top2_random': float(top2r.mean()),
               'occl_sig': float(occ.mean()), 'attr_sig': float(att.mean()), 'sig_chance': float(chance_sig.mean()), 'occl_minus_attr': float((occ - att).mean()), 'occl_minus_attr_ci': [float(x) for x in cd],
               'H_E1': bool(gh.mean() >= 0.60 and cg[0] > 1 / 3), 'H_E2': bool(top2.mean() >= 0.65 and c2[0] > 1 / 3), 'H_E3': bool(occ.mean() >= chance_sig.mean() + 0.15 and cd[0] > -0.03)}
    tab.append({'model': m, 'active TEST events': len(r), 'feature top-1 hit (E2 replication)': top1.mean(), 'Spearman (E2 replication)': rho.mean(), 'group hit': gh.mean(), 'group chance': 1 / 3, 'group random': ghr.mean(), 'group CI low': cg[0], 'top-2 agreement': top2.mean(), 'top-2 random': top2r.mean(), 'top-2 CI low': c2[0],
                'what-if top-1 in signature': occ.mean(), 'attribution top-1 in signature': att.mean(), 'signature chance': chance_sig.mean(), 'diff CI low': cd[0]})
TB = pd.DataFrame(tab); TB.to_csv(OUT/'e3_faithfulness_table.csv', index=False); say('--- explanations on model-active TEST events (Sep and Oct) ---'); say(TB.round(3).T.to_string())
fr = []
for m in [MODELS[1], MODELS[2]]:
    r = R[R.model == m]
    for f in sorted(r.family.unique()):
        q = r[r.family == f]; sg = SIGN[f]; fr.append({'model': m, 'family': B.FAMS[int(f)], 'events': len(q), 'group hit': float(np.mean([np.argmax(grp(a)) == np.argmax(d) for a, d in zip(q.attr, q.DG)])), 'top-2 agreement': float(np.mean([np.argmax(a) in np.argsort(-d)[:2] for a, d in zip(q.attr, q.D)])),
                   'what-if top-1 in signature': float(np.mean([FEATS[int(np.argmax(d))] in sg for d in q.D])), 'attribution top-1 in signature': float(np.mean([FEATS[int(np.argmax(a))] in sg for a in q.attr])), 'signature chance': len(sg) / 6})
FF = pd.DataFrame(fr); FF.to_csv(OUT/'e3_family_table.csv', index=False); say(''); say('--- per family ---'); say(FF.round(2).to_string(index=False))
say(''); say('=== pre-registered hypotheses (protocol E3 v1; saved TEST scores, recomputed explanations) ===')
HYP = {}
for m, tag, nm in [(MODELS[1], 'x', 'XGBoost TreeSHAP'), (MODELS[2], 't', 'Transformer gradient x input')]:
    o = OUTR.get(m)
    if o is None: say(f'{nm}: fewer than 10 model-active events -> H-E1..3 FAIL'); HYP.update({f'H-E1{tag}': False, f'H-E2{tag}': False, f'H-E3{tag}': False}); continue
    say(f'H-E1{tag} {nm}: group hit >= 0.60 and lower bound > 1/3: {o["group_hit"]:.3f} ({o["group_ci"][0]:.3f}..{o["group_ci"][1]:.3f}), random {o["group_random"]:.3f} -> {"HOLDS" if o["H_E1"] else "FAILS"}')
    say(f'H-E2{tag} {nm}: top-1 within the two largest drops >= 0.65 and lower bound > 1/3: {o["top2"]:.3f} ({o["top2_ci"][0]:.3f}..{o["top2_ci"][1]:.3f}), random {o["top2_random"]:.3f} -> {"HOLDS" if o["H_E2"] else "FAILS"}')
    say(f'H-E3{tag} {nm}: what-if top-1 in the injected signature >= chance + 0.15 and not worse than the attribution by > 0.03: {o["occl_sig"]:.3f} (chance {o["sig_chance"]:.3f}, attribution {o["attr_sig"]:.3f}, difference {o["occl_minus_attr"]:+.3f} [{o["occl_minus_attr_ci"][0]:+.3f}..{o["occl_minus_attr_ci"][1]:+.3f}]) -> {"HOLDS" if o["H_E3"] else "FAILS"}')
    HYP.update({f'H-E1{tag}': o['H_E1'], f'H-E2{tag}': o['H_E2'], f'H-E3{tag}': o['H_E3']})
HYP['H-E4'] = True; say('H-E4 exactness: SHAP additivity, saved-score parity (XGBoost 1e-3, Transformer 5e-3), gradient first-order check, deletion and group-deletion column checks passed in T01-T08 -> HOLDS')
say(f'{sum(HYP.values())} of {len(HYP)} hold. Failed hypotheses are reported as failed; nothing was retuned. Explanations describe the model score, not the cause of a market event.')
PRODUCT = {m: {'group_reason': OUTR[m]['H_E1'], 'what_if_column': OUTR[m]['H_E3']} for m in OUTR if OUTR[m]}
DECL = {'status': 'DECLARED: product rules of protocol E3 v1 applied to the results of notebook 55', 'protocol_e3_v1_sha256': PHE, 'hypotheses': HYP, 'results': OUTR, 'product': PRODUCT, 'groups': GROUPS3, 'deletion': {'replacement': 0.0, 'bars': DELLEN}, 'mock': MOCK}
(OUT/'declared_explain_v3.json').write_text(json.dumps(DECL, indent=1, default=str)); (OUT/'e3_share_block.txt').write_text('\n'.join(LINES))
say('Saved: declared_explain_v3.json, e3_faithfulness_table.csv, e3_family_table.csv, e3_share_block.txt in models_b2_v1/ . Send the share block to Claude.')


--- explanations on model-active TEST events (Sep and Oct) ---
                                             0               1
model                               XGBoost v2  Transformer v2
active TEST events                         431             412
feature top-1 hit (E2 replication)       0.269           0.505
Spearman (E2 replication)                0.206           0.257
group hit                                0.346           0.614
group chance                             0.333           0.333
group random                             0.346           0.347
group CI low                             0.299           0.566
top-2 agreement                           0.39           0.672
top-2 random                             0.355            0.35
top-2 CI low                             0.343           0.626
what-if top-1 in signature                0.61           0.782
attribution top-1 in signature           0.954           0.847
signature chance                         0.618         